# UCS420: Cognitive Computing
## Assignment 6 — NumPy-II


**Name:** Prashant Narayan Verma  
**Roll No.:** 1024160140

## Q1. Sensor readings

In [1]:
import numpy as np

temperature = np.array([25, 28, 31, 35, 38, 27, 33, 40])
print("Original readings:", temperature)

Original readings: [25 28 31 35 38 27 33 40]


**a) Add the 2 °C error to every reading** (vectorised, no loop)

In [2]:
corrected = temperature + 2
print("Corrected readings:", corrected)

Corrected readings: [27 30 33 37 40 29 35 42]


**b) Celsius → Fahrenheit**, F = (9/5)·C + 32 (applied to the corrected readings)

In [3]:
fahrenheit = (9 / 5) * corrected + 32
print("Fahrenheit:", fahrenheit)

Fahrenheit: [ 80.6  86.   91.4  98.6 104.   84.2  95.  107.6]


**c) Readings greater than 32 °C** (Boolean indexing on the corrected readings)

In [4]:
mask = corrected > 32
print("Boolean mask      :", mask)
print("Readings > 32 °C  :", corrected[mask])

Boolean mask      : [False False  True  True  True False  True  True]
Readings > 32 °C  : [33 37 40 35 42]


**d) How many readings exceed 32 °C?**

In [5]:
print("Count:", np.sum(mask))      # True counts as 1, False as 0

Count: 5


**e) Why vectorisation and Boolean indexing are more efficient**

- *Vectorisation* applies one operation to the whole array at once. The loop runs inside NumPy's compiled C code instead of the Python interpreter, so it avoids per-element interpreter overhead and type checks, and it uses contiguous memory efficiently.
- *Boolean indexing* filters with a single comparison (`corrected > 32`) and a single selection (`corrected[mask]`) instead of a `for` / `if` / `append` loop. Counting is just `np.sum(mask)`.
- The code is also shorter and easier to read, and the speed advantage grows with the size of the data.

## Q2. Daily steps of four users over three days

In [6]:
steps = np.array([
    [5000, 6200, 7100],
    [8000, 7500, 9000],
    [4500, 5100, 4800],
    [9000, 8500, 9500]])
print(steps)

[[5000 6200 7100]
 [8000 7500 9000]
 [4500 5100 4800]
 [9000 8500 9500]]


In [7]:
print("a) Total steps          :", steps.sum())
print("b) Mean steps           :", round(steps.mean(), 2))
print("c) Maximum / Minimum    :", steps.max(), "/", steps.min())
print("d) Total per day  (axis=0):", steps.sum(axis=0))
print("e) Total per user (axis=1):", steps.sum(axis=1))

flat_index = steps.argmax()
user, day = np.unravel_index(flat_index, steps.shape)
print("f) argmax (flattened index):", flat_index)
print(f"   Position (row, col) = ({user}, {day})  ->  User {user + 1}, Day {day + 1}, {steps[user, day]} steps")

a) Total steps          : 84200
b) Mean steps           : 7016.67
c) Maximum / Minimum    : 9500 / 4500
d) Total per day  (axis=0): [26500 27300 30400]
e) Total per user (axis=1): [18300 24500 14400 27000]
f) argmax (flattened index): 11
   Position (row, col) = (3, 2)  ->  User 4, Day 3, 9500 steps


## Q3. Views, copies, slicing and reshaping

In [8]:
# a) original array
original = np.array([1, 2, 3, 4, 5, 6])
print("a) original:", original)

# b) slice from index 1 to 4  (a VIEW of original)
subset = original[1:5]
print("b) subset  :", subset)

# c) change the first element of subset
subset[0] = 999
print("c) original:", original)
print("   subset  :", subset, " -> original changed too: a slice is a view")

# d) slice with .copy()
copied = original[1:5].copy()
copied[0] = 500
print("d) original:", original)
print("   copied  :", copied, " -> original unaffected: .copy() makes independent data")

a) original: [1 2 3 4 5 6]
b) subset  : [2 3 4 5]
c) original: [  1 999   3   4   5   6]
   subset  : [999   3   4   5]  -> original changed too: a slice is a view
d) original: [  1 999   3   4   5   6]
   copied  : [500   3   4   5]  -> original unaffected: .copy() makes independent data


In [9]:
# e) 1..12 as a 3 x 4 matrix
matrix = np.arange(1, 13).reshape(3, 4)
print("e) matrix:\n", matrix)

# f) indexing and slicing
print("f) First row               :", matrix[0])
print("   Last row                :", matrix[-1])
print("   Second column           :", matrix[:, 1])
print("   Rows 1-2, columns 2-3   :\n", matrix[0:2, 1:3])

e) matrix:
 [[ 1  2  3  4]
 [ 5  6  7  8]
 [ 9 10 11 12]]
f) First row               : [1 2 3 4]
   Last row                : [ 9 10 11 12]
   Second column           : [ 2  6 10]
   Rows 1-2, columns 2-3   :
 [[2 3]
 [6 7]]


*Note for (f):* the question says “rows 1–2 and columns 2–3”, which I read as the 1st–2nd rows and 2nd–3rd columns (counting from 1), i.e. `matrix[0:2, 1:3]`. If rows and columns are counted from 0, the slice would be `matrix[1:3, 2:4]`; both are shown below.

In [10]:
print("1-based reading  matrix[0:2, 1:3]:\n", matrix[0:2, 1:3])
print("0-based reading  matrix[1:3, 2:4]:\n", matrix[1:3, 2:4])

1-based reading  matrix[0:2, 1:3]:
 [[2 3]
 [6 7]]
0-based reading  matrix[1:3, 2:4]:
 [[ 7  8]
 [11 12]]


In [11]:
# g) flatten() vs ravel()
flat_copy = matrix.flatten()
flat_view = matrix.ravel()
print("g) flatten():", flat_copy)
print("   ravel()  :", flat_view)
print("   flatten shares memory with matrix?", np.shares_memory(matrix, flat_copy))
print("   ravel   shares memory with matrix?", np.shares_memory(matrix, flat_view))

# h) modify the ravel() result
flat_view[0] = 100
print("\nh) After flat_view[0] = 100, matrix is:\n", matrix)
print("   -> the original changed: ravel() returns a view (when memory is contiguous)")

# i) modify the flatten() result
flat_copy[1] = 700
print("\ni) After flat_copy[1] = 700, matrix is:\n", matrix)
print("   -> the original is unchanged: flatten() always returns a copy")

# j) attributes
print("\nj) shape:", matrix.shape, "| ndim:", matrix.ndim, "| size:", matrix.size, "| dtype:", matrix.dtype)

g) flatten(): [ 1  2  3  4  5  6  7  8  9 10 11 12]
   ravel()  : [ 1  2  3  4  5  6  7  8  9 10 11 12]
   flatten shares memory with matrix? False
   ravel   shares memory with matrix? True

h) After flat_view[0] = 100, matrix is:
 [[100   2   3   4]
 [  5   6   7   8]
 [  9  10  11  12]]
   -> the original changed: ravel() returns a view (when memory is contiguous)

i) After flat_copy[1] = 700, matrix is:
 [[100   2   3   4]
 [  5   6   7   8]
 [  9  10  11  12]]
   -> the original is unchanged: flatten() always returns a copy

j) shape: (3, 4) | ndim: 2 | size: 12 | dtype: int64


## Q4. Predicting an assistance score with Ordinary Least Squares

In [12]:
X = np.array([
    [6, 70, 3],
    [5, 50, 6],
    [8, 80, 2],
    [4, 30, 8]])
y = np.array([40, 65, 30, 85])

# a) shape and dimensions
print("a) X.shape:", X.shape, "| X.ndim:", X.ndim, "| X.size:", X.size)
print("   -> 4 users (rows) x 3 features (columns: sleep hours, activity level, stress level)")

a) X.shape: (4, 3) | X.ndim: 2 | X.size: 12
   -> 4 users (rows) x 3 features (columns: sleep hours, activity level, stress level)


In [13]:
# b) transpose
Xt = X.T
print("b) X.T:\n", Xt, "\n   shape:", Xt.shape)

b) X.T:
 [[ 6  5  8  4]
 [70 50 80 30]
 [ 3  6  2  8]] 
   shape: (3, 4)


**Meaning of the transpose:** `X.T` swaps rows and columns. In `X` each row is a user and each column a feature; in `X.T` each row is one feature measured across all four users. Multiplying `X.T` by `X` or by `y` therefore combines each feature across users, which is exactly what the least-squares equations need.

In [14]:
# c) X.T @ X   (3x3 matrix of feature-feature sums of products)
XtX = Xt @ X
print("c) X.T @ X:\n", XtX)

# d) inverse - valid only if the determinant is non-zero
print("\n   det(X.T @ X) =", round(np.linalg.det(XtX), 2), "(non-zero, so the inverse exists)")
XtX_inv = np.linalg.inv(XtX)
print("d) inv(X.T @ X):\n", XtX_inv)

c) X.T @ X:
 [[  141  1430    96]
 [ 1430 14700   910]
 [   96   910   113]]

   det(X.T @ X) = 753700.0 (non-zero, so the inverse exists)
d) inv(X.T @ X):
 [[ 1.10521428 -0.09848746 -0.14581398]
 [-0.09848746  0.00891203  0.01190129]
 [-0.14581398  0.01190129  0.0368847 ]]


In [15]:
# e) OLS:  beta = (X^T X)^-1 X^T y
Xty = Xt @ y
beta = XtX_inv @ Xty
print("e) X.T @ y :", Xty)
print("   beta    :", beta)

# sanity check with NumPy's built-in least-squares solver
beta_check, *_ = np.linalg.lstsq(X, y, rcond=None)
print("   lstsq   :", beta_check, "| same result:", np.allclose(beta, beta_check))

e) X.T @ y : [ 1145 11000  1250]
   beta    : [-0.15921454  0.14083853 10.06302242]
   lstsq   : [-0.15921454  0.14083853 10.06302242] | same result: True


**f) What the coefficients mean:** `beta = [β_sleep, β_activity, β_stress]`. Each coefficient is the change in the predicted assistance score for a one-unit increase in that feature while the other two stay fixed.

- **Sleep hours (≈ −0.16):** almost no effect on the score in this data; the small negative sign means slightly less assistance with more sleep, but it is close to zero.
- **Activity level (≈ +0.14):** a small positive effect; each extra activity point raises the predicted score by about 0.14.
- **Stress level (≈ +10.06):** by far the strongest factor; each extra stress point raises the predicted assistance score by about 10, so higher stress means more assistance is predicted.

The model has no intercept term and only four data points, so it is a toy illustration and not a reliable predictor.

In [16]:
# g) predict for a new user
new_user = np.array([5, 40, 7])
predicted = new_user @ beta
print("g) Predicted assistance score for", new_user, "=", round(predicted, 2))

# fitted values vs actual, to see how well the model matches the training data
print("   Fitted values:", np.round(X @ beta, 2), " Actual:", y)

g) Predicted assistance score for [ 5 40  7] = 75.28
   Fitted values: [39.09 66.62 30.12 84.09]  Actual: [40 65 30 85]
